# TGLF FILTER $\times$ NBASIS vs GS2 on the four `kbm_free` KBM databases (Fusion_PhD-qp5.34)

Advisory numbers only. Does TGLF's `FILTER` change the Gaussian width that `FIND_WIDTH` selects, and what does that do to $\gamma$, $\omega$ and the selected mode?

- Data: `TGLF/Runs/LATIN_HYPERCUBE/<case>/kbm_free/fw_{f2p0,f0p0,fm1p5}_nb{2-4,4-8,16-16}`, ALL 1000 samples each, built from the GS2 `kbm_free` decks by GyroRun (`results/tglf_filter_width/*.yaml`).
  `FILTER` 2.0 = frequency filter (TGLF default), 0.0 = off, $-1.5$ = Najlaoui odd-parity filter (`tglf_LS.f90:284`). `FIND_WIDTH` T, TGLF default width grid, `IBRANCH` $-1$, `USE_BPER`/`USE_BPAR` from each deck.
- Selected width: pyro `gaussian_width` of the TGLF output.
- Population: GS2-KBM = user classifier (`mode_classification.classify`) with $\gamma_{\rm GS2}>0$ and `growth_rate_tolerance` $<0.05$ (the qp5.24 gate); counts at $0.01$ also printed.
  $\chi_e/\chi_i$ uses deck-consistent species (ions share ion1 $a/L_T$, every $a/L_n$ is the electron's, as GyroRun): `kbm_free` `pyroscan.json` stores no derived settings, so `sample_pyro` alone returns the template ion2 gradient. The labels with and without the ties are compared.
- Bias statistic as qp5.30: $r=\log_{10}(\gamma_{\rm TGLF}/\gamma_{\rm GS2})$ over cases with TGLF $\gamma>0$; TGLF misses ($\gamma=0$) are counted and kept in LogMed $\lvert\gamma-\gamma_{\rm GS2}\rvert$.

In [ ]:
import os
for threads in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[threads] = "1"       # one BLAS thread per worker process (qp5.33 oversubscription)
import json, re, tempfile, warnings
from multiprocessing import get_context
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import pyrokinetics
from pyrokinetics import Pyro, PyroHypercube
from general_analysis import mode_classification as mc

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
print("pyrokinetics", pyrokinetics.__version__, pyrokinetics.__file__)
data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))   # the WORK tree: the GS2 kbm_free databases
tglf_root = Path(os.path.expandvars("$GYRO_DATA_OUTPUT"))   # the new TGLF leaves are on scratch only
cases = ["SPR-045", "M1", "R4", "R3"]
filters = {"f2p0": 2.0, "f0p0": 0.0, "fm1p5": -1.5}
nbases = [(2, 4), (4, 8), (16, 16)]
tolerance_gate, tolerance_tight = 0.05, 0.01
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
rng = np.random.default_rng(0)
analysis_name = "tglf_filter_width"

In [ ]:
def tie_species(pyro):
    # GyroRun's links for the kbm_free cubes (not stored in pyroscan.json): ions share the first ion's a/L_T, every a/L_n is the electron's
    species = pyro.local_species
    ions = [name for name in species.names if name != "electron"]
    for ion in ions[1:]:
        species[ion].inverse_lt = species[ions[0]].inverse_lt
    for ion in ions:
        species[ion].inverse_ln = species["electron"].inverse_ln
    species.enforce_quasineutrality("electron")


def species_numbers(pyro):
    return {name: (float(pyro.local_species[name].temp.m), float(pyro.local_species[name].dens.m), float(pyro.local_species[name].inverse_lt.m))
            for name in pyro.local_species.names}


def species_untied_and_tied(sample):
    pyro = scan_in_process.sample_pyro(sample)
    untied = species_numbers(pyro)
    tie_species(pyro)
    return untied, species_numbers(pyro)


def deck_species(directory):
    # the deck GS2 ran is stored in input.out.nc as input_file (the WORK tree keeps outputs only)
    text = xr.open_dataset(directory / "input.out.nc").input_file.values.tobytes().decode(errors="ignore").replace("\x00", "")
    deck = Path(tempfile.mkdtemp()) / "input.in"
    deck.write_text(text)
    return species_numbers(Pyro(gk_file=deck))


def bias_stats(gamma_tglf, gamma_gs2, n_boot=10000):
    found = gamma_tglf > 0
    ratio_log = np.log10(gamma_tglf[found] / gamma_gs2[found])
    boot = [np.median(rng.choice(ratio_log, ratio_log.size)) for _ in range(n_boot)]
    return dict(n=gamma_tglf.size, n_found=int(found.sum()), n_miss=int((~found).sum()),
                frac_above=float((ratio_log > 0).mean()), median_log_ratio=float(np.median(ratio_log)),
                ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),
                mean_ratio=float((gamma_tglf[found] / gamma_gs2[found]).mean()),
                logmed_abs_err=float(np.log10(np.median(np.abs(gamma_tglf - gamma_gs2)))))

## Per case: GS2 population, deck-consistency check, TGLF arms

One database per loop iteration (pyro's `enforce_consistent_beta_prime` is state-dependent across scans created in one process, qp5.33; nothing here uses $\beta'$, but the habit is kept).

In [ ]:
gs2, tglf, population_rows, tie_rows, check_rows = {}, {}, [], [], []
for case in cases:
    leaf_gs2 = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf_gs2 / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf_gs2 / "pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data
    names = np.array([str(name) for name in gs2_output.sample_name.values])
    assert list(names) == list(gs2_scan.pyro_dict)
    scan_in_process = gs2_scan

    # species per sample, untied (template ion2) and tied (as the deck), in worker processes
    with get_context("fork").Pool(n_cpu) as pool:
        pairs = pool.map(species_untied_and_tied, range(len(names)), chunksize=8)
    untied, tied = [pair[0] for pair in pairs], [pair[1] for pair in pairs]
    # deck check: the tied species equal each run's own deck, to 1e-6, for samples spread over the cube
    for sample in (0, 1, 7, len(names) // 3, len(names) // 2, len(names) - 1):
        deck = deck_species(leaf_gs2 / names[sample])
        error = max(abs(tied[sample][s][k] - deck[s][k]) / max(abs(deck[s][k]), 1e-12) for s in deck for k in range(3))
        check_rows.append(dict(case=case, sample=names[sample], max_rel_species_error_tied=error,
                               max_rel_species_error_untied=max(abs(untied[sample][s][k] - deck[s][k]) / max(abs(deck[s][k]), 1e-12) for s in deck for k in range(3))))
    assert max(r["max_rel_species_error_tied"] for r in check_rows if r["case"] == case) < 1e-6, f"{case}: tied species differ from the decks"

    heat = gs2_output.heat.sum("field").pint.dequantify()
    particle = gs2_output.particle.sum("field").pint.dequantify()
    def chi(group, species):
        numerator = denominator = 0.0
        for s in group:
            temp, dens, inverse_lt = np.array([sp[s] for sp in species]).T
            numerator = numerator + heat.sel(species=s).values - 1.5 * temp * particle.sel(species=s).values
            denominator = denominator + dens * temp * inverse_lt
        return numerator / denominator
    ions = [str(s) for s in gs2_output.species.values if s != "electron"]
    labels = {}
    with np.errstate(divide="ignore", invalid="ignore"):
        for tie_name, species in (("tied", tied), ("untied", untied)):
            labels[tie_name] = np.asarray(mc.classify(dict(
                T=gs2_output.tearing_parameter.values.ravel(),
                omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                chi_ratio=chi(["electron"], species) / chi(ions, species))))
    gamma = gs2_output.growth_rate.pint.dequantify().values.ravel()
    tolerance = gs2_output.growth_rate_tolerance.pint.dequantify().values.ravel()
    omega = gs2_output.mode_frequency.pint.dequantify().values.ravel()
    positive = np.isfinite(gamma) & (gamma > 0)
    kbm = positive & (tolerance < tolerance_gate) & (labels["tied"] == "KBM")
    kbm_untied = positive & (tolerance < tolerance_gate) & (labels["untied"] == "KBM")
    population_rows.append(dict(case=case, samples=len(names), gamma_positive=int(positive.sum()),
                                kbm_tol005=int(kbm.sum()), kbm_tol001=int((positive & (tolerance < tolerance_tight) & (labels["tied"] == "KBM")).sum()),
                                kbm_untied_tol005=int(kbm_untied.sum()), label_changed_by_ties=int((labels["tied"] != labels["untied"]).sum()),
                                kbm_gained_by_ties=int((kbm & ~kbm_untied).sum()), kbm_lost_by_ties=int((~kbm & kbm_untied).sum())))
    print(population_rows[-1])
    gs2[case] = dict(names=names[kbm], gamma=gamma[kbm], omega=omega[kbm], units=str(gs2_output.growth_rate.data.units))

    # TGLF arms, through the scan
    tglf[case] = {}
    for filter_key in filters:
        for nb_min, nb_max in nbases:
            leaf = tglf_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"fw_{filter_key}_nb{nb_min}-{nb_max}"
            tglf_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
            tglf_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
            tglf_output = tglf_scan.gk_output.data
            assert [str(n) for n in tglf_output.sample_name.values] == list(names)
            selected = tglf_output.set_index(sample="sample_name").sel(sample=gs2[case]["names"])
            assert re.sub(r"_pyroscan_base\d+", "", str(selected.growth_rate.data.units)) == re.sub(r"_pyroscan_base\d+", "", gs2[case]["units"]), "normalisations differ"
            growth = selected.growth_rate.pint.dequantify()
            flags = json.loads((leaf / "manifest.json").read_text())["code_flags"]   # settings GyroRun put on the source deck
            assert flags["IBRANCH"] == -1 and flags["FILTER"] == filters[filter_key] and (flags["NBASIS_MIN"], flags["NBASIS_MAX"]) == (nb_min, nb_max)
            assert (growth.fillna(-np.inf).argmax("mode") == 0).all()  # mode 0 = dominant
            tglf[case][filter_key, nb_min, nb_max] = dict(
                gamma=growth.isel(mode=0).clip(min=0).values,
                omega=selected.mode_frequency.pint.dequantify().isel(mode=0).values,
                width=selected.gaussian_width.pint.dequantify().values)
    print(case, "TGLF arms loaded:", len(tglf[case]))
population = pd.DataFrame(population_rows)
print(population.to_string())
print(pd.DataFrame(check_rows).to_string())

## Bias table: one row per case $\times$ FILTER $\times$ NBASIS

In [ ]:
rows = []
for case in cases:
    for (filter_key, nb_min, nb_max), arm in tglf[case].items():
        rows.append(dict(case=case, filter=filters[filter_key], nbasis_min=nb_min, nbasis_max=nb_max, **bias_stats(arm["gamma"], gs2[case]["gamma"])))
summary = pd.DataFrame(rows)
print(summary.round(3).to_string())

## Scatter: one figure per case, rows FILTER 2 / 0 / $-1.5$, columns NBASIS (2,4) / (4,8) / (16,16)

Format copied from qp5.30 (`tglf_search_bias_vs_gs2.ipynb`, branch `analysis/tglf-search-bias` @ `76791ab`).

In [ ]:
Path("plots").mkdir(exist_ok=True)
colour = {"SPR-045": "tab:blue", "M1": "tab:orange", "R4": "tab:green", "R3": "tab:red"}
for case in cases:
    fig, axes = plt.subplots(3, 3, figsize=(15, 15), sharex=True, sharey=True)
    for ax, ((filter_key, nb_min, nb_max), arm) in zip(axes.ravel(), tglf[case].items()):
        found = arm["gamma"] > 0
        ax.scatter(gs2[case]["gamma"][found], arm["gamma"][found], s=18, alpha=0.7, color=colour[case], label=f"{case} (n={found.sum()})")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_title(f"FILTER {filters[filter_key]:g}, NBASIS {nb_min}-{nb_max}", fontsize=11)
        stats = summary[(summary.case == case) & (summary["filter"] == filters[filter_key]) & (summary.nbasis_min == nb_min)].iloc[0]
        ax.text(0.97, 0.03, f"n={stats.n_found} (+{stats.n_miss} TGLF misses)\nabove $y=x$: {100*stats.frac_above:.0f}%\n"
                f"median $\\log_{{10}}(\\gamma_T/\\gamma_G)$ = {stats.median_log_ratio:+.2f} [{stats.ci_lo:+.2f}, {stats.ci_hi:+.2f}]\n"
                f"mean ratio {stats.mean_ratio:.2f}\nLogMed $|\\gamma$ err$|$ = {stats.logmed_abs_err:.2f}",
                transform=ax.transAxes, ha="right", va="bottom", fontsize=10, bbox=dict(fc="white", alpha=0.85))
    for ax in axes[:, 0]: ax.set_ylabel(r"$\gamma_{\rm TGLF}$")
    for ax in axes[-1]: ax.set_xlabel(r"$\gamma_{\rm GS2}$")
    low = min(min(ax.get_xlim()[0], ax.get_ylim()[0]) for ax in axes.ravel()); high = max(max(ax.get_xlim()[1], ax.get_ylim()[1]) for ax in axes.ravel())
    for ax in axes.ravel():
        ax.plot([low, high], [low, high], "k--", lw=1); ax.set_xlim(low, high); ax.set_ylim(low, high); ax.set_aspect("equal")
    axes[0, 0].legend(loc="upper left")
    fig.savefig(f"plots/fig_gamma_vs_gs2_{case}.png", dpi=200, bbox_inches="tight")
    plt.show()

## Does FILTER change the selected width? (GS2-KBMs, per case and NBASIS)

Pairs are the same sample at two FILTER values. Changed = $\lvert W_{\rm a}/W_{\rm b}-1\rvert$ above 5% (50%). Mode changed = the sign of $\omega$ flips, or $\lvert\omega\rvert$ moves by a factor of 2 (pyro's $\omega$ convention for both arms). TGLF misses ($\gamma=0$) are dropped from a pair.
Compared pairs: FILTER 0 vs 2 (the question) and FILTER $-1.5$ vs 0.

In [ ]:
width_rows = []
for case in cases:
    for nb_min, nb_max in nbases:
        for (key_a, key_b) in (("f0p0", "f2p0"), ("fm1p5", "f0p0")):
            arm_a, arm_b = tglf[case][key_a, nb_min, nb_max], tglf[case][key_b, nb_min, nb_max]
            both = (arm_a["gamma"] > 0) & (arm_b["gamma"] > 0)
            ratio = arm_a["width"][both] / arm_b["width"][both]
            omega_a, omega_b = arm_a["omega"][both], arm_b["omega"][both]
            mode_changed = (np.sign(omega_a) != np.sign(omega_b)) | (np.abs(omega_a) > 2 * np.abs(omega_b)) | (np.abs(omega_b) > 2 * np.abs(omega_a))
            width_b = arm_b["width"][both]
            width_rows.append(dict(
                case=case, nbasis_min=nb_min, nbasis_max=nb_max, filter_a=filters[key_a], filter_b=filters[key_b], n_pairs=int(both.sum()),
                frac_width_change_gt5pct=float((np.abs(ratio - 1) > 0.05).mean()), frac_width_change_gt50pct=float((np.abs(ratio - 1) > 0.5).mean()),
                median_width_ratio=float(np.median(ratio)), frac_mode_changed=float(mode_changed.mean()),
                median_width_b_changed=float(np.median(width_b[mode_changed])) if mode_changed.any() else np.nan,
                median_width_b_unchanged=float(np.median(width_b[~mode_changed])),
                median_abs_omega_b_changed=float(np.median(np.abs(omega_b[mode_changed]))) if mode_changed.any() else np.nan,
                median_abs_omega_b_unchanged=float(np.median(np.abs(omega_b[~mode_changed])))))
width_table = pd.DataFrame(width_rows)
print(width_table.round(3).to_string())
for case in cases:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
    for ax, (nb_min, nb_max) in zip(axes, nbases):
        ax.scatter(tglf[case]["f2p0", nb_min, nb_max]["width"], tglf[case]["f0p0", nb_min, nb_max]["width"], s=18, alpha=0.7, color=colour[case])
        ax.set_title(f"{case}, NBASIS {nb_min}-{nb_max}", fontsize=11)
        ax.set_xlabel(r"$W_{\rm selected}$, FILTER 2"); ax.plot([0, 3], [0, 3], "k--", lw=1)
    axes[0].set_ylabel(r"$W_{\rm selected}$, FILTER 0")
    fig.savefig(f"plots/fig_width_filter0_vs_filter2_{case}.png", dpi=200, bbox_inches="tight")
    plt.show()

## Omega vs GS2 (both arms)

Median $\log_{10}(\lvert\omega_{\rm TGLF}\rvert/\lvert\omega_{\rm GS2}\rvert)$ with bootstrap 95% CI, and the fraction with the same sign of $\omega$ (pyro's convention for both codes), over cases with TGLF $\gamma>0$.

In [ ]:
omega_rows = []
for case in cases:
    for (filter_key, nb_min, nb_max), arm in tglf[case].items():
        found = arm["gamma"] > 0
        ratio_log = np.log10(np.abs(arm["omega"][found]) / np.abs(gs2[case]["omega"][found]))
        ratio_log = ratio_log[np.isfinite(ratio_log)]
        boot = [np.median(rng.choice(ratio_log, ratio_log.size)) for _ in range(10000)]
        omega_rows.append(dict(case=case, filter=filters[filter_key], nbasis_min=nb_min, nbasis_max=nb_max, n=int(ratio_log.size),
                               median_log_abs_omega_ratio=float(np.median(ratio_log)), ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),
                               frac_same_sign=float((np.sign(arm["omega"][found]) == np.sign(gs2[case]["omega"][found])).mean())))
omega_table = pd.DataFrame(omega_rows)
print(omega_table.round(3).to_string())

In [ ]:
Path("csv").mkdir(exist_ok=True)
summary.to_csv("csv/bias_by_case_setting.csv", index=False)
width_table.to_csv("csv/width_change_by_case_setting.csv", index=False)
omega_table.to_csv("csv/omega_vs_gs2_by_case_setting.csv", index=False)
population.to_csv("csv/kbm_population.csv", index=False)
pd.DataFrame(check_rows).to_csv("csv/species_tie_deck_check.csv", index=False)
print(sorted(p.name for p in Path(".").rglob("*") if p.is_file()))